In [ ]:
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
baseline = pypsa.Network(
    "../../resources/network/Historical_2015_reduced_solved.nc"
)

print("Snapshots:", len(baseline.snapshots))
print("Start:", baseline.snapshots[0])
print("End:", baseline.snapshots[-1])

In [ ]:
bess_case = baseline.copy()

In [ ]:
bess_case.add(
    "StorageUnit",
    "Research_BESS_Beauly",
    bus="Beauly",
    carrier="Battery",
    
    # Power capacity
    p_nom=100,
    
    # 100 MW × 2 hours = 200 MWh
    max_hours=2,
    
    # Battery efficiencies
    efficiency_store=0.92,
    efficiency_dispatch=0.92,
    
    # 0.1% loss per hour
    standing_loss=0.001,
    
    # Start empty
    state_of_charge_initial=0,
    
    # Fixed-size battery
    p_nom_extendable=False,
    
    # Small operating cost
    marginal_cost=0
)

In [ ]:
state_of_charge_initial=0

In [ ]:
bess_case.storage_units.loc["Research_BESS_Beauly"]

In [ ]:
print(
    "Battery energy capacity:",
    bess_case.storage_units.loc[
        "Research_BESS_Beauly", "p_nom"
    ]
    *
    bess_case.storage_units.loc[
        "Research_BESS_Beauly", "max_hours"
    ],
    "MWh"
)

In [ ]:
status, condition = bess_case.optimize(
    solver_name="highs"
)

print("Status:", status)
print("Condition:", condition)

In [ ]:
bess_name = "Research_BESS_Beauly"

bess_charge = bess_case.storage_units_t.p_store[bess_name]
bess_discharge = bess_case.storage_units_t.p_dispatch[bess_name]
bess_soc = bess_case.storage_units_t.state_of_charge[bess_name]

print("Total charging:", round(bess_charge.sum(), 1), "MWh")
print("Total discharging:", round(bess_discharge.sum(), 1), "MWh")
print("Maximum SOC:", round(bess_soc.max(), 1), "MWh")
print("Final SOC:", round(bess_soc.iloc[-1], 1), "MWh")

In [ ]:
def calculate_beauly_curtailment(network):

    wind = network.generators[
        (network.generators["bus"] == "Beauly")
        & (network.generators["carrier"] == "wind_onshore")
    ]

    wind_names = wind.index

    available = (
        network.generators_t.p_max_pu[wind_names]
        .mul(wind["p_nom"], axis=1)
        .sum(axis=1)
    )

    dispatched = (
        network.generators_t.p[wind_names]
        .sum(axis=1)
    )

    curtailed = (available - dispatched).clip(lower=0)
    curtailed[curtailed < 1e-6] = 0

    return {
        "available_MWh": available.sum(),
        "dispatched_MWh": dispatched.sum(),
        "curtailed_MWh": curtailed.sum(),
        "curtailment_rate_pct":
            100 * curtailed.sum() / available.sum()
    }


baseline_result = calculate_beauly_curtailment(baseline)
bess_result = calculate_beauly_curtailment(bess_case)

comparison = pd.DataFrame({
    "Baseline": baseline_result,
    "100MW_200MWh_BESS": bess_result
})

comparison

In [ ]:
curtailment_reduction = (
    baseline_result["curtailed_MWh"]
    - bess_result["curtailed_MWh"]
)

reduction_pct = (
    100
    * curtailment_reduction
    / baseline_result["curtailed_MWh"]
)

print(
    "Curtailment reduction:",
    round(curtailment_reduction, 1),
    "MWh"
)

print(
    "Curtailment reduction:",
    round(reduction_pct, 2),
    "%"
)

In [ ]:
# Create a control case with NO additional BESS
control_case = pypsa.Network(
    "../../resources/network/Historical_2015_reduced_solved.nc"
)

# Re-optimise using EXACTLY the same method used for our BESS case
status, condition = control_case.optimize(
    solver_name="highs"
)

print("Status:", status)
print("Condition:", condition)

In [ ]:
control_result = calculate_beauly_curtailment(control_case)

check = pd.DataFrame({
    "Original_PyPSA_GB": baseline_result,
    "Reoptimised_Control": control_result,
    "100MW_200MWh_BESS": bess_result
})

check

In [ ]:
from pathlib import Path

network_files = list(
    Path("../../resources/network").glob(
        "Historical_2015_reduced*.nc"
    )
)

for file in network_files:
    print(file.name)

## 5. Controlled BESS Counterfactual

Both the baseline and BESS scenarios are constructed from the same fully assembled
pre-solve PyPSA-GB network. This ensures that differences in results arise from the
addition of storage rather than from differences in optimisation workflow.

In [ ]:
import pypsa
import pandas as pd
import numpy as np

input_network_path = (
    "../../resources/network/"
    "Historical_2015_reduced_network_demand_renewables_"
    "thermal_generators_storage_hydrogen_interconnectors.nc"
)

prepared_network = pypsa.Network(input_network_path)

print("Snapshots:", len(prepared_network.snapshots))
print("Buses:", len(prepared_network.buses))
print("Generators:", len(prepared_network.generators))
print("Storage units:", len(prepared_network.storage_units))
print("Links:", len(prepared_network.links))

In [ ]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nSearching for solve_network.py...")

for path in Path("../..").rglob("solve_network.py"):
    print(path.resolve())

In [ ]:
# Keep only the same 7-day period used in our original experiment
prepared_network.set_snapshots(
    prepared_network.snapshots[
        (prepared_network.snapshots >= "2015-01-01 00:00:00")
        & (prepared_network.snapshots <= "2015-01-07 23:00:00")
    ]
)

print("Snapshots after trimming:", len(prepared_network.snapshots))
print("Start:", prepared_network.snapshots[0])
print("End:", prepared_network.snapshots[-1])

In [ ]:
import inspect
import scripts.solve.solve_network as solve_module

functions = [
    name
    for name, obj in inspect.getmembers(solve_module, inspect.isfunction)
]

print(functions)

In [ ]:
from pathlib import Path

solve_file = Path("../../scripts/solve/solve_network.py")

lines = solve_file.read_text(encoding="utf-8").splitlines()

# Show the final 120 lines of the actual PyPSA-GB solver script
for i, line in enumerate(lines[-120:], start=len(lines)-119):
    print(f"{i}: {line}")

In [ ]:
# Find the exact optimisation call inside PyPSA-GB
for i, line in enumerate(lines, start=1):
    if ".optimize(" in line:
        start = max(1, i - 15)
        end = min(len(lines), i + 25)

        print(f"\n--- Optimisation call around line {i} ---\n")

        for j in range(start, end + 1):
            print(f"{j}: {lines[j-1]}")

In [ ]:
# Find where scenario_config is created in the actual PyPSA-GB solver

for i, line in enumerate(lines, start=1):
    if "scenario_config =" in line:
        print(f"\n--- scenario_config around line {i} ---\n")

        start = max(1, i - 12)
        end = min(len(lines), i + 15)

        for j in range(start, end + 1):
            print(f"{j}: {lines[j-1]}")

In [ ]:
# Show only the important setup calls made before network.optimize()

important_terms = [
    "validate_network_costs",
    "apply_transmission_relaxation",
    "apply_line_rating_overrides",
    "apply_load_shedding_limits",
    "apply_outage_schedule",
    "improve_numerical_conditioning",
    "build_hydro_constraints_callback",
    "_build_neso_boundary_constraints_callback",
    "configure_solver",
    "set_snapshots"
]

for i in range(1791, 1989):
    line = lines[i - 1]

    if any(term in line for term in important_terms):
        print(f"{i}: {line}")

In [ ]:
import yaml
from pathlib import Path

scenario_file = Path("../../config/scenarios.yaml")

with open(scenario_file, "r", encoding="utf-8") as f:
    scenarios = yaml.safe_load(f)

print(type(scenarios))
print(scenarios.keys())

In [ ]:
scenario_config = scenarios["Historical_2015_reduced"].copy()

# Add scenario ID because the solver script expects it
scenario_config["scenario_id"] = "Historical_2015_reduced"

print(
    yaml.safe_dump(
        scenario_config,
        sort_keys=False
    )
)

In [ ]:
import logging
from scripts.solve import solve_network as solve_mod

# Simple logger for our research notebook
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("project1_bess")


def solve_research_case(input_network, scenario_config):
    """
    Solve a research case using the same main preprocessing
    and optimisation steps used by PyPSA-GB.
    """

    # Always work on a fresh copy
    n = input_network.copy()

    # Same preprocessing sequence as PyPSA-GB
    solve_mod.validate_network_costs(n, logger)

    solve_mod.apply_transmission_relaxation(
        n, scenario_config, logger
    )

    solve_mod.apply_line_rating_overrides(
        n, scenario_config, logger
    )

    solve_mod.apply_outage_schedule(
        n, scenario_config, logger
    )

    solve_mod.improve_numerical_conditioning(
        n, logger
    )

    solve_mod.apply_load_shedding_limits(
        n, logger
    )

    # Configure HiGHS
    solver_name, solver_options = solve_mod.configure_solver(
        n,
        "highs",
        {"threads": 4},
        logger
    )

    # PyPSA-GB additional constraints
    solve_mod.log_hydro_constraint_setup(
        n, scenario_config, logger
    )

    hydro_callback = (
        solve_mod.build_hydro_constraints_callback(
            n, scenario_config
        )
    )

    neso_callback = (
        solve_mod._build_neso_boundary_constraints_callback(
            n, scenario_config, logger
        )
    )

    extra_functionality = (
        solve_mod.combine_extra_functionalities(
            hydro_callback,
            neso_callback
        )
    )

    # Solve
    status, condition = n.optimize(
        solver_name=solver_name,
        solver_options=solver_options,
        extra_functionality=extra_functionality
    )

    print("Status:", status)
    print("Condition:", condition)
    print("Objective:", n.objective)

    return n

In [ ]:
baseline_clean = solve_research_case(
    prepared_network,
    scenario_config
)

In [ ]:
baseline_clean_result = calculate_beauly_curtailment(
    baseline_clean
)

baseline_clean_result

In [ ]:
scenario_names = [
    "Historical_2015_reduced",
    "Historical_2017_reduced",
    "Historical_2019_reduced",
    "Historical_2020_reduced",
    "Historical_2023_etys",
    "Historical_2024_fullyear",
]

for name in scenario_names:
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print(
        yaml.safe_dump(
            scenarios[name],
            sort_keys=False
        )
    )

In [ ]:
import pypsa

jan_2015 = pypsa.Network(
    "../../resources/network/Research_2015_Jan_solved.nc"
)

print("Scenario:", jan_2015.name)
print("Snapshots:", len(jan_2015.snapshots))
print("Start:", jan_2015.snapshots[0])
print("End:", jan_2015.snapshots[-1])
print("Objective:", jan_2015.objective)

In [ ]:
# Find all wind-related generator carriers in the January model

wind_carriers = [
    carrier
    for carrier in jan_2015.generators["carrier"].dropna().unique()
    if "wind" in carrier.lower()
]

print("Wind carriers found:")
print(wind_carriers)

print("\nInstalled wind capacity by carrier:")

wind_capacity = (
    jan_2015.generators[
        jan_2015.generators["carrier"].isin(wind_carriers)
    ]
    .groupby("carrier")["p_nom"]
    .sum()
    .sort_values(ascending=False)
)

print(wind_capacity)

In [ ]:
# Select every wind generator in the GB model
gb_wind = jan_2015.generators[
    jan_2015.generators["carrier"].isin(wind_carriers)
]

gb_wind_names = gb_wind.index

# Maximum wind that could have generated each hour
gb_wind_available = (
    jan_2015.generators_t.p_max_pu[gb_wind_names]
    .mul(gb_wind["p_nom"], axis=1)
    .sum(axis=1)
)

# Wind actually dispatched by the optimisation
gb_wind_dispatch = (
    jan_2015.generators_t.p[gb_wind_names]
    .sum(axis=1)
)

# Implied curtailment
gb_wind_curtailment = (
    gb_wind_available - gb_wind_dispatch
).clip(lower=0)

# Remove tiny floating-point numerical noise
gb_wind_curtailment[
    gb_wind_curtailment < 1e-6
] = 0

# January totals
print(
    "GB wind available:",
    round(gb_wind_available.sum() / 1000, 2),
    "GWh"
)

print(
    "GB wind dispatched:",
    round(gb_wind_dispatch.sum() / 1000, 2),
    "GWh"
)

print(
    "GB wind curtailed:",
    round(gb_wind_curtailment.sum() / 1000, 2),
    "GWh"
)

print(
    "GB wind curtailment rate:",
    round(
        100
        * gb_wind_curtailment.sum()
        / gb_wind_available.sum(),
        2
    ),
    "%"
)

In [ ]:
# January 2015 wind results by carrier

wind_results_by_carrier = []

for carrier in wind_carriers:

    names = jan_2015.generators.index[
        jan_2015.generators["carrier"] == carrier
    ]

    capacity_mw = jan_2015.generators.loc[names, "p_nom"].sum()

    available = (
        jan_2015.generators_t.p_max_pu[names]
        .mul(jan_2015.generators.loc[names, "p_nom"], axis=1)
        .sum(axis=1)
    )

    dispatched = jan_2015.generators_t.p[names].sum(axis=1)

    curtailed = (available - dispatched).clip(lower=0)
    curtailed[curtailed < 1e-6] = 0

    available_gwh = available.sum() / 1000
    dispatched_gwh = dispatched.sum() / 1000
    curtailed_gwh = curtailed.sum() / 1000

    curtailment_pct = (
        100 * curtailed.sum() / available.sum()
        if available.sum() > 0
        else 0
    )

    wind_results_by_carrier.append({
        "carrier": carrier,
        "capacity_MW": capacity_mw,
        "available_GWh": available_gwh,
        "dispatched_GWh": dispatched_gwh,
        "curtailed_GWh": curtailed_gwh,
        "curtailment_pct": curtailment_pct
    })

wind_results_by_carrier = pd.DataFrame(wind_results_by_carrier)

print(
    wind_results_by_carrier.round({
        "capacity_MW": 2,
        "available_GWh": 2,
        "dispatched_GWh": 2,
        "curtailed_GWh": 3,
        "curtailment_pct": 3
    })
)

In [ ]:
# Transmission line congestion statistics - January 2015

line_stats = []

for line in jan_2015.lines.index:

    s_nom = jan_2015.lines.at[line, "s_nom"]
    s_max_pu = jan_2015.lines.at[line, "s_max_pu"]

    limit = s_nom * s_max_pu

    # Skip lines with unusable/zero limits
    if limit <= 0:
        continue

    flow = jan_2015.lines_t.p0[line].abs()

    loading_pct = 100 * flow / limit

    line_stats.append({
        "line": line,
        "bus0": jan_2015.lines.at[line, "bus0"],
        "bus1": jan_2015.lines.at[line, "bus1"],
        "limit": limit,
        "max_loading_pct": loading_pct.max(),
        "avg_loading_pct": loading_pct.mean(),
        "hours_90": (loading_pct >= 90).sum(),
        "hours_95": (loading_pct >= 95).sum(),
        "hours_99": (loading_pct >= 99).sum(),
        "zero_r": jan_2015.lines.at[line, "r"] == 0
    })

line_stats = pd.DataFrame(line_stats)

line_stats = line_stats.sort_values(
    ["hours_99", "max_loading_pct"],
    ascending=False
)

print(
    line_stats.head(15).round({
        "limit": 2,
        "max_loading_pct": 2,
        "avg_loading_pct": 2
    })
)

In [ ]:
# Rank January 2015 onshore wind curtailment by bus

onshore_names = jan_2015.generators.index[
    jan_2015.generators["carrier"] == "wind_onshore"
]

# Hourly available generation for each onshore wind generator
onshore_available_by_gen = (
    jan_2015.generators_t.p_max_pu[onshore_names]
    .mul(
        jan_2015.generators.loc[onshore_names, "p_nom"],
        axis=1
    )
)

# Hourly dispatched generation
onshore_dispatch_by_gen = (
    jan_2015.generators_t.p[onshore_names]
)

# Curtailment by generator
onshore_curtailment_by_gen = (
    onshore_available_by_gen - onshore_dispatch_by_gen
).clip(lower=0)

onshore_curtailment_by_gen[
    onshore_curtailment_by_gen < 1e-6
] = 0

# Total curtailment by generator
curtailment_by_gen = onshore_curtailment_by_gen.sum()

# Map each generator to its bus
curtailment_by_bus = (
    pd.DataFrame({
        "bus": jan_2015.generators.loc[
            curtailment_by_gen.index, "bus"
        ],
        "curtailment_MWh": curtailment_by_gen.values
    })
    .groupby("bus")["curtailment_MWh"]
    .sum()
    .sort_values(ascending=False)
)

curtailment_by_bus = curtailment_by_bus[
    curtailment_by_bus > 1e-6
]

print("Onshore wind curtailment by bus:")
print((curtailment_by_bus / 1000).round(4).rename("curtailment_GWh"))

print(
    "\nTotal:",
    round(curtailment_by_bus.sum() / 1000, 4),
    "GWh"
)

In [ ]:
# ---------------------------------------------------------
# Temporal overlap:
# Beauly wind curtailment vs Beauly-Errochty congestion
# ---------------------------------------------------------

# Onshore wind generators located at Beauly
beauly_wind_names = jan_2015.generators.index[
    (jan_2015.generators["carrier"] == "wind_onshore") &
    (jan_2015.generators["bus"] == "Beauly")
]

# Hourly available Beauly wind
beauly_wind_available = (
    jan_2015.generators_t.p_max_pu[beauly_wind_names]
    .mul(
        jan_2015.generators.loc[beauly_wind_names, "p_nom"],
        axis=1
    )
    .sum(axis=1)
)

# Hourly dispatched Beauly wind
beauly_wind_dispatch = (
    jan_2015.generators_t.p[beauly_wind_names]
    .sum(axis=1)
)

# Hourly Beauly wind curtailment
beauly_wind_curtailment = (
    beauly_wind_available - beauly_wind_dispatch
).clip(lower=0)

beauly_wind_curtailment[
    beauly_wind_curtailment < 1e-6
] = 0


# Beauly -> Errochty line 1 loading
line = "1"

line_limit = (
    jan_2015.lines.at[line, "s_nom"]
    * jan_2015.lines.at[line, "s_max_pu"]
)

beauly_errochty_loading = (
    jan_2015.lines_t.p0[line].abs()
    / line_limit
    * 100
)

# Define binding hours
binding_99 = beauly_errochty_loading >= 99

# Curtailment statistics
total_curtailment = beauly_wind_curtailment.sum()

curtailment_binding = (
    beauly_wind_curtailment[binding_99].sum()
)

curtailment_nonbinding = (
    beauly_wind_curtailment[~binding_99].sum()
)

curtailment_hours = (
    beauly_wind_curtailment > 1e-6
).sum()

curtailment_hours_binding = (
    (beauly_wind_curtailment > 1e-6)
    & binding_99
).sum()

print("Beauly wind curtailment:", round(total_curtailment, 2), "MWh")
print("Binding hours >=99%:", binding_99.sum())
print("Hours with curtailment:", curtailment_hours)
print("Curtailment hours while >=99%:", curtailment_hours_binding)

print(
    "Curtailment during >=99% loading:",
    round(curtailment_binding, 2),
    "MWh"
)

print(
    "Curtailment outside >=99% loading:",
    round(curtailment_nonbinding, 2),
    "MWh"
)

print(
    "Share of curtailment during >=99% loading:",
    round(
        100 * curtailment_binding / total_curtailment,
        2
    ),
    "%"
)

In [ ]:
# Inspect the exact hours when Beauly wind was curtailed

curtailment_mask = beauly_wind_curtailment > 1e-6

event_table = pd.DataFrame({
    "wind_available_MW": beauly_wind_available,
    "wind_dispatch_MW": beauly_wind_dispatch,
    "wind_curtailment_MW": beauly_wind_curtailment,

    "line1_loading_pct":
        jan_2015.lines_t.p0["1"].abs()
        / (
            jan_2015.lines.at["1", "s_nom"]
            * jan_2015.lines.at["1", "s_max_pu"]
        ) * 100,

    "line3_loading_pct":
        jan_2015.lines_t.p0["3"].abs()
        / (
            jan_2015.lines.at["3", "s_nom"]
            * jan_2015.lines.at["3", "s_max_pu"]
        ) * 100,

    "line0_Beauly_Peterhead_MW":
        jan_2015.lines_t.p0["0"],

    "line2_Beauly_Peterhead_MW":
        jan_2015.lines_t.p0["2"]
})

event_table = event_table[curtailment_mask]

print(event_table.round(2))

In [ ]:
import pypsa
import pandas as pd

# Load both solved models
jan_only = pypsa.Network(
    "../../resources/network/Research_2015_Jan_solved.nc"
)

jan_plus7 = pypsa.Network(
    "../../resources/network/Research_2015_JanPlus7_solved.nc"
)

print("Jan-only snapshots:", len(jan_only.snapshots))
print("Jan+7 snapshots:", len(jan_plus7.snapshots))


def beauly_metrics(n):

    # Beauly onshore wind generators
    names = n.generators.index[
        (n.generators["carrier"] == "wind_onshore") &
        (n.generators["bus"] == "Beauly")
    ]

    available = (
        n.generators_t.p_max_pu[names]
        .mul(n.generators.loc[names, "p_nom"], axis=1)
        .sum(axis=1)
    )

    dispatched = n.generators_t.p[names].sum(axis=1)

    curtailment = (available - dispatched).clip(lower=0)
    curtailment[curtailment < 1e-6] = 0

    # Beauly -> Errochty line 1
    line = "1"

    limit = (
        n.lines.at[line, "s_nom"]
        * n.lines.at[line, "s_max_pu"]
    )

    loading = (
        n.lines_t.p0[line].abs()
        / limit
        * 100
    )

    return pd.DataFrame({
        "available_MW": available,
        "dispatch_MW": dispatched,
        "curtailment_MW": curtailment,
        "line1_loading_pct": loading
    })


old = beauly_metrics(jan_only)
extended = beauly_metrics(jan_plus7)

# Compare ONLY 31 January
old_31 = old.loc["2015-01-31"]
extended_31 = extended.loc["2015-01-31"]

print("\n--- JANUARY-ONLY MODEL ---")
print("31 Jan curtailment:",
      round(old_31["curtailment_MW"].sum(), 2), "MWh")
print("Curtailment hours:",
      (old_31["curtailment_MW"] > 1e-6).sum())
print("Hours line >=99%:",
      (old_31["line1_loading_pct"] >= 99).sum())

print("\n--- JANUARY + 7 DAYS MODEL ---")
print("31 Jan curtailment:",
      round(extended_31["curtailment_MW"].sum(), 2), "MWh")
print("Curtailment hours:",
      (extended_31["curtailment_MW"] > 1e-6).sum())
print("Hours line >=99%:",
      (extended_31["line1_loading_pct"] >= 99).sum())

In [ ]:
# Compare TOTAL January Beauly curtailment
# between the two optimisation horizons

old_january = old.loc[
    "2015-01-01 00:00:00":"2015-01-31 23:00:00"
]

extended_january = extended.loc[
    "2015-01-01 00:00:00":"2015-01-31 23:00:00"
]

print(
    "Jan-only total Beauly curtailment:",
    round(old_january["curtailment_MW"].sum(), 2),
    "MWh"
)

print(
    "Jan+7 total Beauly curtailment:",
    round(extended_january["curtailment_MW"].sum(), 2),
    "MWh"
)

print(
    "Jan-only curtailment hours:",
    (old_january["curtailment_MW"] > 1e-6).sum()
)

print(
    "Jan+7 curtailment hours:",
    (extended_january["curtailment_MW"] > 1e-6).sum()
)

In [ ]:
import inspect

print(
    inspect.signature(
        jan_plus7.optimize.optimize_with_rolling_horizon
    )
)

In [ ]:
print(
    hasattr(
        jan_plus7.optimize,
        "optimize_with_rolling_horizon"
    )
)

In [ ]:
import inspect

from scripts.solve import solve_network as sn

functions_to_check = [
    "configure_solver",
    "build_hydro_constraints_callback",
    "_build_neso_boundary_constraints_callback",
    "combine_extra_functionalities",
]

for name in functions_to_check:
    func = getattr(sn, name)

    print("\n", "=" * 70)
    print(name)
    print(inspect.signature(func))

In [ ]:
import inspect

print(inspect.getsource(sn.configure_solver))

In [ ]:
print(inspect.getsource(sn.combine_extra_functionalities))

In [ ]:
preprocessing_functions = [
    "validate_network_costs",
    "apply_transmission_relaxation",
    "apply_line_rating_overrides",
    "apply_outage_schedule",
    "improve_numerical_conditioning",
    "apply_load_shedding_limits",
]

for name in preprocessing_functions:
    func = getattr(sn, name)

    print("\n" + "=" * 70)
    print(name)
    print(inspect.signature(func))

In [ ]:
import pypsa
import pandas as pd

# Full-horizon reference
reference = pypsa.Network(
    "../../resources/network/Research_2015_JanPlus7_solved.nc"
)

# New rolling-horizon result
rolling = pypsa.Network(
    "../../resources/network/Research_2015_JanPlus7_rolling_solved.nc"
)


def january_metrics(n):

    jan = n.snapshots[
        (n.snapshots >= "2015-01-01 00:00:00") &
        (n.snapshots <= "2015-01-31 23:00:00")
    ]

    # -----------------------------------
    # Beauly onshore wind
    # -----------------------------------

    wind = n.generators.index[
        (n.generators["carrier"] == "wind_onshore") &
        (n.generators["bus"] == "Beauly")
    ]

    available = (
        n.generators_t.p_max_pu.loc[jan, wind]
        .mul(n.generators.loc[wind, "p_nom"], axis=1)
        .sum(axis=1)
    )

    dispatched = (
        n.generators_t.p.loc[jan, wind]
        .sum(axis=1)
    )

    curtailed = (
        available - dispatched
    ).clip(lower=0)

    curtailed[curtailed < 1e-6] = 0


    # -----------------------------------
    # Beauly -> Errochty line 1
    # -----------------------------------

    line = "1"

    limit = (
        n.lines.at[line, "s_nom"]
        * n.lines.at[line, "s_max_pu"]
    )

    loading = (
        n.lines_t.p0.loc[jan, line].abs()
        / limit
        * 100
    )


    # -----------------------------------
    # Load shedding
    # -----------------------------------

    load_shedding = n.generators.index[
        n.generators["carrier"] == "load_shedding"
    ]

    load_shedding_mwh = (
        n.generators_t.p.loc[jan, load_shedding]
        .sum()
        .sum()
    )


    return {
        "Beauly curtailment MWh":
            curtailed.sum(),

        "Beauly curtailment hours":
            (curtailed > 1e-6).sum(),

        "Beauly-Errochty >=99% hours":
            (loading >= 99).sum(),

        "Load shedding MWh":
            load_shedding_mwh
    }


comparison = pd.DataFrame({
    "Full Jan+7": january_metrics(reference),
    "Rolling": january_metrics(rolling)
})

print(comparison.round(2))